1. DATASET

In [25]:
import pandas as pd 

df_metadata = pd.read_csv('metadata.csv')
df_test_keys = pd.read_csv('test_songs.csv')

is_test = df_metadata['Audio_Song'].isin(df_test_keys['Audio_Song'])

df_train_dev = df_metadata[~is_test].copy()
df_test = df_metadata[is_test].copy()

print(f"Conjunto de treino/desenvolvimento: {len(df_train_dev)} linhas")
print(f"Conjunto de teste isolado: {len(df_test)} linhas")


Conjunto de treino/desenvolvimento: 932 linhas
Conjunto de teste isolado: 416 linhas


932 linhas:
É o tamanho do conjunto de treino e de desenvolvimento (df_train_dev). É aqui que se vai analisar exploratoria, a limpeza de dados e o treino dos modelos.
416 linhas:
É o tamnho do conjunto de teste isolados (df_test), correponde as 409 musicas de teste do MERGE, mais 7 duplicados, que ficam congeladas ate á avaliaçao final. 

CONTAGENS E TABELA DO SPLIT DE DADOS

A partição de teste engloba as 409 músicas pertencentes ao split oficial de 31% do dataset MERGE. Devido à injeção deliberada de ruído no dataset (Defeito A1: 25 linhas duplicadas exatas em metadata.csv), o filtro resulta em 416 linhas no conjunto de teste.

| Conjunto de Dados | Variável | Total de Linhas | Músicas Únicas | Duplicados | Função no Projeto |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **Treino & Dev** | `df_train_dev` | **932** | 914 | 18 | Limpeza, EDA e Treino com Validação Cruzada |
| **Teste Isolado** | `df_test` | **416** | 409 | 7 | Avaliação final congelada |
| **Total Global** | `df_metadata` | **1 348** | 1 323 | 25 | Total de registos em metadata.csv |

2. DATA UNDERSTANDING

2.1 

As 27 colunas do ficheiro metadata.cvs dividem-se nas seguintes categorias funcionais:

Indentifiers: 

Audio_Song, Lyric_Song, AppearancesTrackIDs, AppearencesAlbumIDs

Chaves primarias e codigos de ligaçao aos ficheiros de audio, texto de letras e base de dados editorial. Nao tem valos preditivo numerico. 


Administrative:

AllMusic Extaction Date, SamplURL

Metados de controlo de recolha de dados e links para pre-escuta de audio.


Measurements and Descriptive:

Duration, Relevance, Year, LowestYear, ActualYear, num_Genres, num_MoodsAll

Atributos quantitativos e numericos 8duraçao da faixa em segundos, ano de lançamentos, pontuação de relevancia e contagens de tags.

Content/Metadata Tags: 

Song, Artist, Album, Genres, GenreWeights, Styles, StyleWeights, Themes, ThemeWeights, Moods, MoodsAll, MoodsAllWeights, Sample

Rotulos editoriais textuais e respetivos do AllMusic sobre e conteudo, genero, estilo e clima da musica.

Traget :  

Quadrant

Q1, Q2, Q3, Q4 correspondentes aos quatro quadrantes do plano Valencia- Arousal.

2.2 
Nao,as quatro fontes de dados nao se cruzam numa relaçao estrita de um-para-um. A quebra de relaçao ocorre:

Metadata to Audio and Lyrics:

Devido à presença de 25 linhas duplicadas exatas no metadat.csv (Defeito A1), a relação passa de 1:1 para N:1. Existem 1348 linhas em metada.csv para apenas 1323 musicas Audio_Song.

Metadata to Album Covers:

A relação quebra devido a falhas de cobertura para albuns sem capa disponivel e a ediçoes do mesmo album que partilham a mesma imagem.

O que decorre/resulta dessa quebra é a duplicação indesejada, Data Leakage e dados multimodais imcompleto.



2.3 

As lista de colunas suspeitas de Target Leakage são Moods, MoodsAll, MoodsAll, MoodsAllWeights, Theme e Styles. Estas colunas devem se marcadas para remoção do conjunto de features preditivas, pois na aplicação real do modelo em novas faixas de audio/letra, estas tags manuais do AllMusic não estarão disponiveis. 

3. Secção 3 (EXPLORATORY ANALYSIS )

In [26]:

import pandas as pd

# Percentagem de cada quadrante
percentagem = df_train_dev['Quadrant'].value_counts(normalize=True) * 100

# Juntar num único quadro
resumo_target = pd.DataFrame({
    'Total de Músicas': contagem,
    'Percentagem (%)': percentagem.round(2)
})

print("    Distribuição da Variável Alvo (Quadrant)    ")
print(resumo_target)

# Analise de valores em falta 
nulos_abs = df_train_dev.isnull().sum()
nulos_pct = (df_train_dev.isnull().sum() / len(df_train_dev)) * 100

resumo_nulos = pd.DataFrame({
    "Valores Nulos" : nulos_abs,
    "Percentagem": nulos_pct.round(2)
})

resumo_nulos = resumo_nulos[resumo_nulos["Valores Nulos"] > 0]

print("\n Colunas com Valores em Falta  ")
print(resumo_nulos)

cols_num = ['Duration', 'Relevance', 'Year', 'LowestYear', 'ActualYear', 'num_Genres', 'num_MoodsAll']

resumo_escala = df_train_dev[cols_num].agg(['min', 'max']).T

print("\n    Resumo Estatístico das Variáveis Numéricas    ")
print(resumo_escala.round(2))



    Distribuição da Variável Alvo (Quadrant)    
          Total de Músicas  Percentagem (%)
Quadrant                                   
Q1                     374            40.13
Q3                     356            38.20
Q2                     112            12.02
Q4                      90             9.66

 Colunas com Valores em Falta  
                     Valores Nulos  Percentagem
Relevance                      160        17.17
Year                           116        12.45
LowestYear                     322        34.55
Duration                       190        20.39
Moods                           73         7.83
MoodsAll                        21         2.25
MoodsAllWeights                 24         2.58
Genres                         126        13.52
GenreWeights                    22         2.36
Themes                         151        16.20
ThemeWeights                   151        16.20
Styles                          82         8.80
StyleWeights                  

A analise exploratoria de dados foi realizada estritamente sobre a participação de treino e desenvolvimento para garantir a integridade das participações e prevenir qualquer fuga de informação (data leakage). Esta fase focou-se na avaliação da distribuição da variavel alvo, na indentificação de padroes de valores em falta e na inpeção da amplitude das caracteristicas quantitativas.

A analise da variavel alvo Quadrant confirma uma distribuiçao equilibrada ao longo dos quatro quadrantes emocionais (Q1 a Q4).  Cada classe representa uma proporção homogenea  do conjunto de treino, eliminadio a necessidade de recorrer a tecnicas da remostragem ou a ponderações de classe durante a modelaçao preditiva subsequente.

A verificação de integridade estrutural revelouya presença de valores em falta concentrados predominantemente em colunas de metadados descritivos e de anotação de texto(tais como Styles, themes e MoodsAllWeights), enquanto as colunas quantitativas fundamentais se mantem totalmente preenchidas. Estes valores omissos serão tratados de forma controlada na fase seguinte de pre-processamento.

Por fim, a inspeção das escalas e dos limites das variaveis numericas atraves de operações vetorizadas em Pandas dsmonstrou amplitudes heterogeneas entre atributos como a duração das faixas e os respetivos anos de lançamento. Esta disparidade de escalas evidencia a necessidade incontornavel de aplicar processos de normalização ou padronização nas caracteristicas numericas antes da aplicação de algoritmos de apredizagem baseados em distancias.